In [9]:
import torch
import decipher
import numpy as np
import scanpy as sc

# --- 1. Load everything ---
adata = sc.read_h5ad("decipher_outputs/decipher_run_full.h5ad")
X = adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X
V_np = np.load("uv_leading_metrics_improved/V_pred_learned.npy") # Ensure this is your gene-space velocity

# --- 2. Load Model ---
# --- 2. Load Model using the official helper ---
# The loader expects the run_id to be in adata.uns
# Based on your path, your run_id is '2026-04-25-02-35-20-buoyant-rosy-tostada'
adata.uns["decipher"]["run_id"] = "2026-04-25-02-35-20-buoyant-rosy-tostada"

# Call the built-in loader
decipher_model = decipher.tl._decipher.data.decipher_load_model(adata)
decipher_model.eval()

print("Model loaded successfully using built-in helper.")

# --- 3. Run Option A (Jacobian) ---
X_torch = torch.tensor(X, dtype=torch.float32)

def encoder_jacobian_velocity(model, X_torch, V_np, batch_size=64):
    V_z = np.zeros((X_torch.shape[0], 10), dtype=np.float32)
    
    for i in range(0, X_torch.shape[0], batch_size):
        # Prepare batch
        xb = X_torch[i:i+batch_size].clone().requires_grad_(True)
        
        # -------------------------------------------------------
        # Use the correct attribute found: encoder_x_to_z
        # -------------------------------------------------------
        out = model.encoder_x_to_z(xb)
        
        # Decipher encoders typically return (z_mean, z_scale)
        # We take the mean [0] to track the center of the movement
        zb = out[0] if isinstance(out, tuple) else out
        
        for d in range(zb.shape[1]):
            # Compute gradient of latent dimension 'd' with respect to input genes
            grad = torch.autograd.grad(
                zb[:, d].sum(), 
                xb, 
                retain_graph=(d < zb.shape[1]-1)
            )[0]
            
            # Project gene-space velocity onto this gradient
            # V_z = dot(grad, V_gene)
            V_z[i:i+batch_size, d] = (
                grad.detach().cpu().numpy() * V_np[i:i+batch_size]
            ).sum(axis=1)
            
    return V_z

# Run the calculation
V_latent = encoder_jacobian_velocity(decipher_model, X_torch, V_np)

# --- 4. Save the result ---
np.save("decipher_outputs/V_latent_jacobian.npy", V_latent)
print("Jacobian latent velocity saved!")

Model loaded successfully using built-in helper.


ValueError: operands could not be broadcast together with shapes (64,12553) (64,11483) 

In [8]:
# Check what attributes are available in your specific version
print([attr for attr in dir(decipher_model) if not attr.startswith('_')])

['T_destination', 'add_module', 'apply', 'bfloat16', 'buffers', 'call_super_init', 'children', 'compile', 'compute_v_z_numpy', 'config', 'cpu', 'cuda', 'decoder_v_to_z', 'decoder_z_to_x', 'device', 'double', 'dummy_param', 'dump_patches', 'encoder_x_to_z', 'encoder_zx_to_v', 'eval', 'extra_repr', 'float', 'forward', 'get_buffer', 'get_extra_state', 'get_parameter', 'get_submodule', 'guide', 'half', 'impute_gene_expression_numpy', 'ipu', 'load_state_dict', 'model', 'modules', 'mtia', 'named_buffers', 'named_children', 'named_modules', 'named_parameters', 'parameters', 'register_backward_hook', 'register_buffer', 'register_forward_hook', 'register_forward_pre_hook', 'register_full_backward_hook', 'register_full_backward_pre_hook', 'register_load_state_dict_post_hook', 'register_load_state_dict_pre_hook', 'register_module', 'register_parameter', 'register_state_dict_post_hook', 'register_state_dict_pre_hook', 'requires_grad_', 'set_extra_state', 'set_submodule', 'share_memory', 'state_dic

In [10]:
# --- 1. Load everything ---
adata = sc.read_h5ad("decipher_outputs/decipher_run_full.h5ad")
# These are the 12,553 genes Decipher expects
decipher_genes = adata.var_names.tolist()

# Load your RegulatoryFlow outputs
V_pred_raw = np.load("uv_leading_metrics_improved/V_pred_learned.npy") # Ensure this is your gene-space velocity
A_genes = np.load("uv_leading_fixed_metrics/A_genes.npy", allow_pickle=True)

# --- 2. Align Velocity to Decipher Genes ---
print(f"Aligning {len(A_genes)} velocity genes to {len(decipher_genes)} model genes...")

# Create mapping of gene name -> index in the velocity matrix
v_gene_to_idx = {gene: i for i, gene in enumerate(A_genes)}

# Initialize a zero matrix with the shape Decipher expects (Cells, 12553)
V_aligned = np.zeros((V_pred_raw.shape[0], len(decipher_genes)), dtype=np.float32)

# Fill the aligned matrix where genes overlap
found_count = 0
for i, gene in enumerate(decipher_genes):
    if gene in v_gene_to_idx:
        V_aligned[:, i] = V_pred_raw[:, v_gene_to_idx[gene]]
        found_count += 1

print(f"Successfully aligned {found_count} genes. (Remaining will be 0)")

# Now define X and V_np for the Jacobian
X = adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X
V_np = V_aligned # This is now (N, 12553)

# --- 3. Run Jacobian ---
X_torch = torch.tensor(X, dtype=torch.float32)

# Use the attribute we found earlier: encoder_x_to_z
def encoder_jacobian_velocity(model, X_torch, V_np, batch_size=64):
    V_z = np.zeros((X_torch.shape[0], 10), dtype=np.float32)
    for i in range(0, X_torch.shape[0], batch_size):
        xb = X_torch[i:i+batch_size].clone().requires_grad_(True)
        
        # Get latent mean from the specific decipher encoder
        out = model.encoder_x_to_z(xb)
        zb = out[0] if isinstance(out, tuple) else out
        
        for d in range(zb.shape[1]):
            grad = torch.autograd.grad(
                zb[:, d].sum(), xb, retain_graph=(d < zb.shape[1]-1)
            )[0]
            
            # This now works because shapes are both (batch, 12553)
            V_z[i:i+batch_size, d] = (
                grad.detach().cpu().numpy() * V_np[i:i+batch_size]
            ).sum(axis=1)
    return V_z

V_latent = encoder_jacobian_velocity(decipher_model, X_torch, V_np)

# --- 4. Save for the other environment ---
np.save("decipher_outputs/V_latent_jacobian.npy", V_latent)
print("Saved aligned Jacobian velocity.")

Aligning 11483 velocity genes to 12553 model genes...
Successfully aligned 11483 genes. (Remaining will be 0)
Saved aligned Jacobian velocity.
